# Basin delineation

In [ ]:
from andeangc import config as cfg
from andeangc import basin_delineation as bd

# {institution: DEM tag}. The institution names the gauge files (SENAMHI_data.csv)
# and the delineated output (basins_SENAMHI.gpkg); the DEM tag names the per-region
# DEM (DEM_PERU_90m.tif); joined, they name the folder under data/resources
# (SENAMHI_PERU/). Chile and Patagonia are absent on purpose — CAMELS-CL and
# PMET-obs already ship their own basin polygons.
regions = {"SENAMHI": "PERU",
           "SNHI": "ARG"}

wbt = bd.whitebox_tools()   # pass verbose=True to see each tool's output
print(wbt.version())

## Gauge coordinates to pour points

In [ ]:
pour_points = {source: bd.write_pour_points(source, region)
               for source, region in regions.items()}

## Flow routing and delineation

In [ ]:
routing, snapped, basin_rasters = {}, {}, {}

for source, region in regions.items():
    routing[source] = bd.route_flow(region, wbt=wbt)   
    snapped[source], basin_rasters[source] = bd.delineate(
        source, routing[source], pour_points=pour_points[source], wbt=wbt)

## Vectorisation

In [ ]:
basins = {}

for source, region in regions.items():
    basins[source] = bd.basins_to_gdf(basin_rasters[source], snapped[source])
    basins[source].to_file(cfg.RESOURCES / f"{source}_{region}" / f"basins_{source}.gpkg")

{source: len(gdf) for source, gdf in basins.items()}

## Cleanup

In [ ]:
intermediates = (list(pour_points.values()) + list(snapped.values())
                 + [f for rasters in routing.values() for f in rasters.values()]
                 + [f for rasters in basin_rasters.values() for f in rasters])
bd.remove_files(intermediates);